# FerroML vs scikit-learn: A Comprehensive Comparison

This notebook provides a detailed side-by-side comparison of FerroML and scikit-learn. While sklearn is the de facto standard for ML in Python, FerroML offers unique capabilities that make it the better choice for practitioners who need **statistical rigor**.

**What this notebook covers:**
1. API compatibility - FerroML follows sklearn conventions
2. Statistical features sklearn lacks (p-values, CIs, odds ratios)
3. Performance comparison (Rust vs Python)
4. Feature-by-feature comparison
5. When to use which library

In [ ]:
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

## 1. API Compatibility

FerroML was designed to be a drop-in replacement for sklearn. The core API (fit, predict, transform) is identical.

### 1.1 Linear Regression: Identical API

In [ ]:
# Generate sample data
X = np.random.randn(100, 3)
y = 2 * X[:, 0] - X[:, 1] + 0.5 * X[:, 2] + np.random.randn(100) * 0.5

# sklearn
from sklearn.linear_model import LinearRegression as SklearnLR
sklearn_model = SklearnLR()
sklearn_model.fit(X, y)

# FerroML
from ferroml.linear import LinearRegression as FerroMLLR
ferroml_model = FerroMLLR()
ferroml_model.fit(X, y)

print("Both libraries use identical API: fit(X, y), predict(X), coef_, intercept_")
print("\nsklearn coefficients:", sklearn_model.coef_)
print("FerroML coefficients:", ferroml_model.coef_)
print("\nDifference (should be ~0):", np.abs(sklearn_model.coef_ - ferroml_model.coef_).max())

### 1.2 Preprocessing: Same Interface

In [ ]:
# sklearn
from sklearn.preprocessing import StandardScaler as SklearnScaler
sklearn_scaler = SklearnScaler()
X_sklearn = sklearn_scaler.fit_transform(X)

# FerroML
from ferroml.preprocessing import StandardScaler as FerroMLScaler
ferroml_scaler = FerroMLScaler()
X_ferroml = ferroml_scaler.fit_transform(X)

print("Both use fit_transform(X), transform(X), inverse_transform(X)")
print("\nMax difference in scaled data:", np.abs(X_sklearn - X_ferroml).max())

### 1.3 Pipeline: Compatible Structure

In [ ]:
# sklearn pipeline
from sklearn.pipeline import Pipeline as SklearnPipeline
from sklearn.preprocessing import StandardScaler as SkScaler
from sklearn.linear_model import Ridge as SkRidge

sklearn_pipe = SklearnPipeline([
    ('scaler', SkScaler()),
    ('model', SkRidge(alpha=1.0))
])
sklearn_pipe.fit(X, y)

# FerroML pipeline
from ferroml.pipeline import Pipeline as FerroMLPipeline
from ferroml.preprocessing import StandardScaler as FMScaler
from ferroml.linear import RidgeRegression as FMRidge

ferroml_pipe = FerroMLPipeline([
    ('scaler', FMScaler()),
    ('model', FMRidge(alpha=1.0))
])
ferroml_pipe.fit(X, y)

# Compare predictions
X_test = np.random.randn(5, 3)
print("sklearn pipeline predictions:", sklearn_pipe.predict(X_test))
print("FerroML pipeline predictions:", ferroml_pipe.predict(X_test))

## 2. What sklearn CANNOT Do

This is where FerroML shines. sklearn is designed for prediction, not statistical inference. FerroML provides full statistical diagnostics that sklearn simply doesn't have.

### 2.1 Coefficient Significance Testing

In [ ]:
# sklearn: Just gives you coefficients
print("=" * 60)
print("sklearn LinearRegression")
print("=" * 60)
print(f"Coefficients: {sklearn_model.coef_}")
print(f"Intercept: {sklearn_model.intercept_}")
print("\n...and that's ALL you get. No p-values, no standard errors, no CIs.")

In [ ]:
# FerroML: Full statistical inference
print("=" * 60)
print("FerroML LinearRegression")
print("=" * 60)
print(ferroml_model.summary())

In [ ]:
# Programmatic access to statistical inference
ci = ferroml_model.coefficients_with_ci(level=0.95)

print("\nProgrammatic access to coefficient inference:")
print("-" * 60)
for coef_info in ci:
    sig = "***" if coef_info['p_value'] < 0.001 else "**" if coef_info['p_value'] < 0.01 else "*" if coef_info['p_value'] < 0.05 else ""
    print(f"{coef_info['name']:>12}: {coef_info['estimate']:>8.4f} (p={coef_info['p_value']:.4f}) {sig}")

### 2.2 Prediction Intervals

sklearn can only give point predictions. FerroML provides proper prediction intervals that account for both parameter uncertainty and irreducible error.

In [ ]:
X_new = np.array([[1.0, 0.5, -0.5], [0.0, 1.0, 1.0], [-1.0, -0.5, 0.0]])

# sklearn: Just point predictions
sklearn_preds = sklearn_model.predict(X_new)
print("sklearn predictions (no uncertainty quantification):")
for i, pred in enumerate(sklearn_preds):
    print(f"  Sample {i}: {pred:.4f}")

print()

In [ ]:
# FerroML: Predictions WITH intervals
ferroml_preds, lower, upper = ferroml_model.predict_interval(X_new, level=0.95)

print("FerroML predictions with 95% prediction intervals:")
for i, (pred, lo, hi) in enumerate(zip(ferroml_preds, lower, upper)):
    width = hi - lo
    print(f"  Sample {i}: {pred:.4f} [{lo:.4f}, {hi:.4f}] (width: {width:.4f})")

### 2.3 Odds Ratios for Logistic Regression

In many domains (medicine, epidemiology, social sciences), odds ratios are essential for interpretation. sklearn doesn't provide them.

In [ ]:
# Generate binary classification data
n = 500
age = np.random.normal(50, 10, n)  # Age in years
bmi = np.random.normal(25, 5, n)   # BMI
X_class = np.column_stack([age, bmi])

# Create outcome: P(disease) increases with age and BMI
logits = -12 + 0.1 * age + 0.15 * bmi
probs = 1 / (1 + np.exp(-logits))
y_class = np.random.binomial(1, probs).astype(float)

print(f"Disease prevalence: {y_class.mean():.1%}")

In [ ]:
# sklearn LogisticRegression
from sklearn.linear_model import LogisticRegression as SkLogistic

sk_lr = SkLogistic(penalty=None, max_iter=1000)
sk_lr.fit(X_class, y_class)

print("sklearn LogisticRegression:")
print(f"  Coefficients: {sk_lr.coef_[0]}")
print(f"  Intercept: {sk_lr.intercept_[0]}")
print("\n  To get odds ratios, you have to manually compute exp(coef).")
print("  sklearn provides NO confidence intervals for odds ratios.")

In [ ]:
# FerroML LogisticRegression
from ferroml.linear import LogisticRegression as FerroLogistic

fm_lr = FerroLogistic()
fm_lr.fit(X_class, y_class)

print("FerroML LogisticRegression:")
print(fm_lr.summary())

In [ ]:
# Odds ratios with confidence intervals
odds = fm_lr.odds_ratios_with_ci(level=0.95)

feature_names = ['Age (per year)', 'BMI (per unit)']

print("\nOdds Ratios with 95% CI (FerroML exclusive):")
print("-" * 60)
for name, o in zip(feature_names, odds):
    print(f"{name:20} OR = {o['odds_ratio']:.3f} [{o['ci_lower']:.3f}, {o['ci_upper']:.3f}]")

print("\nInterpretation:")
print(f"  - Each additional year of age increases odds of disease by {(odds[0]['odds_ratio']-1)*100:.1f}%")
print(f"  - Each unit increase in BMI increases odds of disease by {(odds[1]['odds_ratio']-1)*100:.1f}%")

### 2.4 Model Fit Statistics

FerroML provides comprehensive model fit diagnostics that sklearn doesn't.

In [ ]:
# sklearn: Limited fit statistics
print("sklearn model fit statistics:")
print(f"  R² (score): {sklearn_model.score(X, y):.4f}")
print("  ...that's all sklearn provides.")
print()

In [ ]:
# FerroML: Comprehensive fit statistics
print("FerroML model fit statistics:")
print(f"  R²:                {ferroml_model.r_squared():.4f}")
print(f"  Adjusted R²:       {ferroml_model.adjusted_r_squared():.4f}")
print(f"  F-statistic:       {ferroml_model.f_statistic():.4f}")
print(f"  F-statistic p-val: {ferroml_model.f_pvalue():.2e}")
print(f"  Residual Std Err:  {ferroml_model.residual_std_error():.4f}")
print(f"  Log-Likelihood:    {ferroml_model.log_likelihood():.2f}")
print(f"  AIC:               {ferroml_model.aic():.2f}")
print(f"  BIC:               {ferroml_model.bic():.2f}")

## 3. Performance Comparison

FerroML is written in Rust, compiled to native code. Let's compare training and prediction speeds.

In [ ]:
def benchmark(func, n_iterations=100):
    """Run function multiple times and return average time in ms."""
    times = []
    for _ in range(n_iterations):
        start = time.perf_counter()
        func()
        end = time.perf_counter()
        times.append((end - start) * 1000)  # Convert to ms
    return np.mean(times), np.std(times)

### 3.1 Linear Regression Performance

In [ ]:
# Generate larger dataset for benchmarking
sizes = [1000, 5000, 10000, 50000]
n_features = 20

print("Linear Regression Training Time (ms)")
print("=" * 60)
print(f"{'n_samples':>10} {'sklearn':>15} {'FerroML':>15} {'Speedup':>12}")
print("-" * 60)

for n in sizes:
    X_bench = np.random.randn(n, n_features)
    y_bench = X_bench @ np.random.randn(n_features) + np.random.randn(n) * 0.5
    
    sk_model = SklearnLR()
    fm_model = FerroMLLR()
    
    sk_time, sk_std = benchmark(lambda: sk_model.fit(X_bench, y_bench), n_iterations=50)
    fm_time, fm_std = benchmark(lambda: fm_model.fit(X_bench, y_bench), n_iterations=50)
    
    speedup = sk_time / fm_time
    print(f"{n:>10} {sk_time:>12.2f}±{sk_std:.2f} {fm_time:>12.2f}±{fm_std:.2f} {speedup:>10.2f}x")

### 3.2 Tree Model Performance

In [ ]:
from sklearn.ensemble import RandomForestClassifier as SkRF
from ferroml.trees import RandomForestClassifier as FerroRF

# Generate classification data
n_bench = 5000
n_features = 20
X_rf = np.random.randn(n_bench, n_features)
y_rf = (X_rf[:, 0] + X_rf[:, 1] > 0).astype(float)

print("Random Forest (100 trees) Training Time")
print("=" * 50)

# sklearn
sk_rf = SkRF(n_estimators=100, max_depth=10, random_state=42, n_jobs=1)
start = time.perf_counter()
sk_rf.fit(X_rf, y_rf)
sk_rf_time = (time.perf_counter() - start) * 1000
print(f"sklearn:  {sk_rf_time:.2f} ms")

# FerroML
fm_rf = FerroRF(n_estimators=100, max_depth=10, random_state=42)
start = time.perf_counter()
fm_rf.fit(X_rf, y_rf)
fm_rf_time = (time.perf_counter() - start) * 1000
print(f"FerroML: {fm_rf_time:.2f} ms")
print(f"\nSpeedup: {sk_rf_time/fm_rf_time:.2f}x")

### 3.3 Prediction Performance

In [ ]:
# Prediction benchmark
X_pred = np.random.randn(10000, n_features)

print("Random Forest Prediction Time (10,000 samples)")
print("=" * 50)

sk_pred_time, _ = benchmark(lambda: sk_rf.predict(X_pred), n_iterations=50)
fm_pred_time, _ = benchmark(lambda: fm_rf.predict(X_pred), n_iterations=50)

print(f"sklearn:  {sk_pred_time:.2f} ms")
print(f"FerroML: {fm_pred_time:.2f} ms")
print(f"Speedup: {sk_pred_time/fm_pred_time:.2f}x")

## 4. Feature-by-Feature Comparison

Let's systematically compare what each library offers.

In [ ]:
comparison_data = [
    ["Feature", "sklearn", "FerroML", "Notes"],
    ["="*30, "="*10, "="*10, "="*40],
    ["LINEAR MODELS", "", "", ""],
    ["LinearRegression", "Yes", "Yes", "Identical API"],
    ["Ridge/Lasso/ElasticNet", "Yes", "Yes", "Identical API"],
    ["LogisticRegression", "Yes", "Yes", "Identical API"],
    ["Coefficient p-values", "No", "Yes", "FerroML exclusive"],
    ["Coefficient CIs", "No", "Yes", "FerroML exclusive"],
    ["Prediction intervals", "No", "Yes", "FerroML exclusive"],
    ["Odds ratios + CI", "No", "Yes", "FerroML exclusive"],
    ["R-style summary()", "No", "Yes", "FerroML exclusive"],
    ["AIC/BIC", "No", "Yes", "FerroML exclusive"],
    ["-"*30, "-"*10, "-"*10, "-"*40],
    ["TREE MODELS", "", "", ""],
    ["DecisionTree", "Yes", "Yes", "Identical API"],
    ["RandomForest", "Yes", "Yes", "Identical API"],
    ["GradientBoosting", "Yes", "Yes", "Identical API"],
    ["HistGradientBoosting", "Yes", "Yes", "LightGBM-style"],
    ["Monotonic constraints", "Yes", "Yes", "Domain knowledge"],
    ["OOB score", "Yes", "Yes", "Out-of-bag"],
    ["Feature importance CI", "No", "Yes", "FerroML via bootstrap"],
    ["-"*30, "-"*10, "-"*10, "-"*40],
    ["PREPROCESSING", "", "", ""],
    ["StandardScaler", "Yes", "Yes", "Identical"],
    ["MinMaxScaler", "Yes", "Yes", "Identical"],
    ["RobustScaler", "Yes", "Yes", "Identical"],
    ["OneHotEncoder", "Yes", "Yes", "Identical"],
    ["SimpleImputer", "Yes", "Yes", "Identical"],
    ["KNNImputer", "Yes", "Yes", "Identical"],
    ["SMOTE", "imblearn", "Yes", "Built-in"],
    ["-"*30, "-"*10, "-"*10, "-"*40],
    ["EXPLAINABILITY", "", "", ""],
    ["Permutation importance", "Yes", "Yes", "With CIs"],
    ["Partial dependence", "Yes", "Yes", "1D and 2D"],
    ["ICE plots", "Yes", "Yes", "With c-ICE"],
    ["SHAP values", "shap lib", "Yes", "TreeSHAP built-in"],
    ["H-statistic", "No", "Yes", "FerroML exclusive"],
    ["-"*30, "-"*10, "-"*10, "-"*40],
    ["AUTOML", "", "", ""],
    ["Auto model selection", "No", "Yes", "Built-in AutoML"],
    ["Time budget", "No", "Yes", "Bandit-based"],
    ["Meta-learning", "No", "Yes", "Warm start"],
    ["Ensemble construction", "No", "Yes", "Auto-sklearn style"],
    ["-"*30, "-"*10, "-"*10, "-"*40],
    ["DEPLOYMENT", "", "", ""],
    ["Pickle serialization", "Yes", "Yes", "Identical"],
    ["ONNX export", "sklearn-onnx", "Yes", "Built-in"],
    ["Rust-native inference", "No", "Yes", "No Python needed"],
    ["Schema validation", "No", "Yes", "Production safety"],
]

for row in comparison_data:
    print(f"{row[0]:<30} {row[1]:<10} {row[2]:<10} {row[3]:<40}")

## 5. Code Comparison: Same Task, Different Capabilities

Let's solve the same problem with both libraries and see the difference in output.

### Task: Build a credit risk model and explain it

In [ ]:
# Generate synthetic credit data
np.random.seed(42)
n_customers = 1000

# Features: income, debt_ratio, credit_history_length, num_late_payments
income = np.random.lognormal(10.5, 0.5, n_customers)  # Annual income
debt_ratio = np.random.beta(2, 5, n_customers)  # Debt to income ratio
credit_history = np.random.exponential(5, n_customers)  # Years of credit history
late_payments = np.random.poisson(1, n_customers)  # Number of late payments

X_credit = np.column_stack([income/10000, debt_ratio*100, credit_history, late_payments])
feature_names = ['Income (10K)', 'Debt Ratio (%)', 'Credit History (yrs)', 'Late Payments']

# Default probability model
logits = -3 - 0.3*(income/10000) + 0.1*(debt_ratio*100) - 0.2*credit_history + 0.8*late_payments
default_prob = 1 / (1 + np.exp(-logits))
y_default = np.random.binomial(1, default_prob).astype(float)

print(f"Dataset: {n_customers} customers")
print(f"Default rate: {y_default.mean():.1%}")

In [ ]:
# sklearn approach
print("=" * 70)
print("sklearn APPROACH")
print("=" * 70)

sk_credit = SkLogistic(penalty=None, max_iter=1000)
sk_credit.fit(X_credit, y_default)

print(f"\nCoefficients: {sk_credit.coef_[0]}")
print(f"Intercept: {sk_credit.intercept_[0]:.4f}")
print(f"\nAccuracy: {sk_credit.score(X_credit, y_default):.4f}")
print("\n--- End of sklearn output ---")
print("\nTo get more info, you need statsmodels or manual calculations.")

In [ ]:
# FerroML approach
print("=" * 70)
print("FerroML APPROACH")
print("=" * 70)

fm_credit = FerroLogistic()
fm_credit.fit(X_credit, y_default)

print(fm_credit.summary())

In [ ]:
# Odds ratios for business interpretation
odds = fm_credit.odds_ratios_with_ci(level=0.95)

print("\nBusiness Interpretation (Odds Ratios):")
print("=" * 70)
for name, o in zip(feature_names, odds):
    or_val = o['odds_ratio']
    if or_val > 1:
        effect = f"increases default odds by {(or_val-1)*100:.1f}%"
    else:
        effect = f"decreases default odds by {(1-or_val)*100:.1f}%"
    sig = "(significant)" if o['ci_lower'] > 1 or o['ci_upper'] < 1 else "(not significant)"
    print(f"  {name}: OR={or_val:.3f} [{o['ci_lower']:.3f}, {o['ci_upper']:.3f}]")
    print(f"    -> Each unit increase {effect} {sig}")
    print()

In [ ]:
# Model comparison metrics
print("Model Quality Metrics:")
print("=" * 50)
print(f"Pseudo R²:           {fm_credit.pseudo_r_squared():.4f}")
print(f"AIC:                 {fm_credit.aic():.2f}")
print(f"BIC:                 {fm_credit.bic():.2f}")

lr_stat, lr_pval = fm_credit.likelihood_ratio_test()
print(f"\nLikelihood Ratio Test:")
print(f"  LR Statistic: {lr_stat:.2f}")
print(f"  p-value:      {lr_pval:.2e}")
if lr_pval < 0.05:
    print("  => Model is significantly better than null model")

## 6. When to Use Which Library

### Use sklearn when:
- You only need predictions, not inference
- You're doing quick prototyping and don't need statistics
- You need algorithms not yet in FerroML (e.g., neural networks)
- You're in a pure Python environment without Rust compilation

### Use FerroML when:
- You need to know if coefficients are **statistically significant**
- You need **prediction intervals** (not just point estimates)
- You need **odds ratios** for interpretation
- You need **model comparison** via AIC/BIC
- You need **R-style summaries** for reports
- You're deploying to production and need **ONNX export** or **Rust inference**
- You want **better performance** from compiled Rust code
- You're doing **regulated ML** where model interpretability is required

## 7. Migration Guide: sklearn to FerroML

Migrating from sklearn to FerroML is straightforward due to API compatibility.

In [ ]:
# Before (sklearn)
print("# BEFORE (sklearn)")
print("""
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

model = LinearRegression()
model.fit(X, y)
predictions = model.predict(X_test)
""")

print("\n# AFTER (FerroML) - Just change imports!")
print("""
from ferroml.linear import LinearRegression, LogisticRegression, RidgeRegression
from ferroml.preprocessing import StandardScaler, OneHotEncoder
from ferroml.pipeline import Pipeline
from ferroml.trees import RandomForestClassifier

model = LinearRegression()
model.fit(X, y)
predictions = model.predict(X_test)

# NEW: Get statistics that sklearn can't provide
print(model.summary())  # R-style output
ci = model.coefficients_with_ci()  # Confidence intervals
pred, lower, upper = model.predict_interval(X_test)  # Prediction intervals
""")

## Summary

| Aspect | sklearn | FerroML |
|--------|---------|----------|
| **API** | Standard | sklearn-compatible |
| **Focus** | Prediction | Prediction + Inference |
| **Statistics** | Minimal | Comprehensive |
| **Performance** | Pure Python | Rust (faster) |
| **Deployment** | Pickle, ONNX (external) | Pickle, ONNX, Rust inference |
| **AutoML** | No | Built-in |

**Bottom line**: If you just need predictions, sklearn works fine. If you need to **understand** and **explain** your models with proper statistics, FerroML is the better choice.

## Next Steps

- **01_getting_started.ipynb**: Introduction to FerroML
- **02_statistical_diagnostics.ipynb**: Deep dive into statistical features
- **04_production_deployment.ipynb**: ONNX export and deployment

For more information, see the [FerroML documentation](https://github.com/ferroml/ferroml).